# ♻️ Smart E-Waste Segregation System: YOLOv8 Training & Live Inference
**Project:** Conveyor-Based Automated Electronic Waste Segregation using YOLOv8 & 6-DOF Robotic Arm  
**GitHub Repository:** [neilgibsonpinto-art/ewaste](https://github.com/neilgibsonpinto-art/ewaste)  
**Target Classes:** `0: pcb`, `1: battery`, `2: display`  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neilgibsonpinto-art/ewaste/blob/main/Smart_EWaste_Colab.ipynb)

## 1. Setup Environment & Clone Project Repository
Clones the repository and installs YOLOv8 and computer vision packages.

In [ ]:
import os
%cd /content
if os.path.exists('/content/ewaste'):
    !rm -rf /content/ewaste

!git clone https://github.com/neilgibsonpinto-art/ewaste.git
%cd /content/ewaste

!pip install -q ultralytics roboflow opencv-python matplotlib Pillow

import torch
import ultralytics
print('PyTorch Version:', torch.__version__)
print('Ultralytics Version:', ultralytics.__version__)
print('GPU Active:', torch.cuda.is_available())

## 2. Dataset Inspection & Preview
Verifies dataset annotations and visualizes sample images.

In [ ]:
import yaml
import glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

data_yaml_path = Path('/content/ewaste/dataset/data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = '/content/ewaste/dataset'
data_config['train'] = 'images/train'
data_config['val'] = 'images/val'

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print('Dataset YAML Verified:')
print(yaml.dump(data_config))

train_images = glob.glob('/content/ewaste/dataset/images/train/*.jpg')[:4]
fig, axes = plt.subplots(1, len(train_images), figsize=(16, 4))
if len(train_images) == 1:
    axes = [axes]
for ax, img_p in zip(axes, train_images):
    ax.imshow(Image.open(img_p))
    ax.set_title(Path(img_p).stem)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train Lightweight YOLOv8-Nano Model
Trains on the 92 e-waste component images using the GPU.

In [ ]:
import os
from ultralytics import YOLO

weights_source = '/content/ewaste/yolov8n.pt' if os.path.exists('/content/ewaste/yolov8n.pt') else 'yolov8n.pt'
model = YOLO(weights_source)

results = model.train(
    data='/content/ewaste/dataset/data.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/ewaste/runs/train',
    name='ewaste_colab_yolov8n',
    plots=True,
    verbose=True
)
print('Training Complete!')

## 4. Visualize Training Performance Curves
Displays training loss, confusion matrix, and Precision-Recall curves.

In [ ]:
from IPython.display import Image as IPyImage, display
import os

train_dir = '/content/ewaste/runs/train/ewaste_colab_yolov8n'
for img_name in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    p = os.path.join(train_dir, img_name)
    if os.path.exists(p):
        print(f'--- {img_name} ---')
        display(IPyImage(p))

## 5. LIVE WEBCAM VIDEO STREAM DETECTION INSIDE COLAB
Streams your laptop webcam directly into Google Colab with real-time YOLOv8 bounding boxes and live object counters!

In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
from PIL import Image
import io
import os
from ultralytics import YOLO

# Load best weights
trained_path = '/content/ewaste/runs/train/ewaste_colab_yolov8n/weights/best.pt'
if not os.path.exists(trained_path):
    trained_path = '/content/ewaste/models/ewaste_yolov8n/weights/best.pt'

print('Loading YOLOv8 model from:', trained_path)
model = YOLO(trained_path)

# Live counts
counts = {'pcb': 0, 'battery': 0, 'display': 0}

# JavaScript WebRTC Bridge for continuous live video inside Colab
def start_live_stream():
    js = Javascript('''
        var video;
        var div;
        var stream;
        var captureCanvas;
        var imgElement;
        var stopBtn;
        var isRunning = true;

        async function init() {
            div = document.createElement('div');
            div.style.cssText = 'padding: 12px; background: #0f172a; border-radius: 12px; display: inline-block; text-align: center;';
            
            stopBtn = document.createElement('button');
            stopBtn.textContent = '⏹ STOP LIVE DETECTION';
            stopBtn.style.cssText = 'padding: 10px 20px; font-weight: bold; background: #ef4444; color: white; border: none; border-radius: 8px; cursor: pointer; margin-bottom: 10px;';
            stopBtn.onclick = () => { isRunning = false; if(stream) stream.getVideoTracks()[0].stop(); div.remove(); };
            div.appendChild(stopBtn);

            imgElement = document.createElement('img');
            imgElement.style.cssText = 'display: block; border-radius: 8px; max-width: 640px; border: 2px solid #38bdf8;';
            div.appendChild(imgElement);
            document.body.appendChild(div);

            video = document.createElement('video');
            stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
            video.srcObject = stream;
            await video.play();

            captureCanvas = document.createElement('canvas');
            captureCanvas.width = 640;
            captureCanvas.height = 480;
        }

        async function getFrame() {
            if (!isRunning) return null;
            var ctx = captureCanvas.getContext('2d');
            ctx.drawImage(video, 0, 0, 640, 480);
            return captureCanvas.toDataURL('image/jpeg', 0.8);
        }

        function showProcessed(dataUrl) {
            if (imgElement) imgElement.src = dataUrl;
        }

        window.colabStream = { init, getFrame, showProcessed };
    ''')
    display(js)
    eval_js('colabStream.init()')

start_live_stream()
print('Live Webcam Stream started! Hold your PCB, Battery, or Display in front of your camera.')

# Process video frames continuously in Python
try:
    while True:
        data_url = eval_js('colabStream.getFrame()')
        if not data_url:
            print('\nStream stopped.')
            break
            
        # Decode base64 to OpenCV image
        header, encoded = data_url.split(',', 1)
        frame_bytes = b64decode(encoded)
        np_arr = np.frombuffer(frame_bytes, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
        
        # Run YOLOv8 detection
        res = model.predict(frame, conf=0.45, verbose=False)[0]
        annotated_frame = res.plot()
        
        # Update counts
        for box in res.boxes:
            cls_name = res.names[int(box.cls[0])]
            if cls_name in counts:
                counts[cls_name] += 1
                
        # Draw top counter bar
        cv2.rectangle(annotated_frame, (0, 0), (640, 40), (15, 23, 42), -1)
        hud_text = f"PCB: {counts['pcb']} | BATTERY: {counts['battery']} | DISPLAY: {counts['display']}"
        cv2.putText(annotated_frame, hud_text, (15, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (56, 189, 248), 2)
        
        # Re-encode and stream back to browser
        _, buffer = cv2.imencode('.jpg', annotated_frame)
        processed_base64 = 'data:image/jpeg;base64,' + b64encode(buffer).decode('utf-8')
        eval_js(f'colabStream.showProcessed("{processed_base64}")')
except Exception as e:
    print('Stream ended:', e)

## 6. Batch Test on Test Images with Static Counter
Runs predictions on dataset test images and prints the final count tally.

In [ ]:
import glob
from collections import Counter
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

trained_path = '/content/ewaste/runs/train/ewaste_colab_yolov8n/weights/best.pt'
if not os.path.exists(trained_path):
    trained_path = '/content/ewaste/models/ewaste_yolov8n/weights/best.pt'

detector = YOLO(trained_path)
val_images = glob.glob('/content/ewaste/dataset/images/val/*.jpg')[:6]
counter = Counter()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_p in zip(axes.flatten(), val_images):
    res = detector.predict(img_p, conf=0.45, verbose=False)[0]
    for box in res.boxes:
        cls_name = res.names[int(box.cls[0])]
        counter[cls_name] += 1
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(Path(img_p).stem)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('=' * 40)
print('  SEGREGATION TALLY SUMMARY')
print('=' * 40)
for k, v in counter.items():
    print(f'  {k.upper()}: {v} item(s)')
print(f'  TOTAL PROCESSED: {sum(counter.values())}')
print('=' * 40)

## 7. Download Trained Model Weights (`best.pt`)
Downloads the `.pt` weights directly to your laptop for edge deployment.

In [ ]:
from google.colab import files
import os

target_file = '/content/ewaste/runs/train/ewaste_colab_yolov8n/weights/best.pt'
if not os.path.exists(target_file):
    target_file = '/content/ewaste/models/ewaste_yolov8n/weights/best.pt'

files.download(target_file)
print('Downloaded weights to your computer!')